WhatNet  –  Japanese

In [ ]:
#  Kuzushiji (くずし字) is classical cursive Japanese script used in pre-Meiji
#  documents.  Kuzushiji-49 covers 49 Hiragana character classes rendered in
#  this historical cursive style.
#    • num_classes   : 49   (Kuzushiji-49: 49 Hiragana classes)
#    • image_size    : 28   (same – Kuzushiji-49 is natively 28×28)
#    • Dataset format: .npz arrays
#                      keys: arr_0 (images, uint8), arr_1 (labels, uint8)
#    • Dataset loader: npz loader
#    • Scaffold branch: 3×3 + 3×1 pair
#                       Kuzushiji characters combine flowing diagonal curves
#                       (captured by 3×3 isotropic) with explicit vertical
#                       stroke endings (captured by 3×1).  The horizontal 1×3
#                       is insufficient because Hiragana strokes are often
#                       diagonal, not axis-aligned.
#    • Augmentation  : random_flip REMOVED entirely – Kuzushiji characters
#                       have asymmetric strokes; flipping creates invalid glyphs
#                       (e.g. mirroring き would not produce any valid kana).
#                       random_rotation ±8° ADDED (via tfa or manual affine) –
#                       cursive documents vary greatly in writing angle, so
#                       mild rotation is the most important generalisation.
#                       NOTE: if tensorflow-addons is unavailable, rotation
#                       falls back to pad-crop only (a warning is printed).
#    • Model name    : WhatNet-Kuzushiji
#
#  Dataset
#  Kuzushiji-49  (Clanuwat et al., 2018  –  arXiv:1812.01718)
#    • 232 365 train + 38 547 test 28×28 greyscale images
#    • 49 classes (balanced-ish Hiragana subset of full Kuzushiji dataset)
#    • Download:
#        https://github.com/rois-codh/kmnist   (official – npz files)
#        https://www.kaggle.com/datasets/anokas/kuzushiji  (Kaggle mirror)
#    • Files expected in CFG['data_dir']:
#        k49-train-imgs.npz  –  arr_0: uint8 (N, 28, 28)
#        k49-train-lbls.npz  –  arr_0: uint8 (N,)
#        k49-test-imgs.npz   –  arr_0: uint8 (M, 28, 28)
#        k49-test-lbls.npz   –  arr_0: uint8 (M,)
#    • Alternative single-file layout (some mirrors):
#        k49_train.npz       –  keys: arr_0 (images), arr_1 (labels)
#        k49_test.npz        –  keys: arr_0 (images), arr_1 (labels)
#    • This script handles BOTH layouts automatically.

In [1]:
!nvidia-smi

Tue May  5 03:36:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:8D:00.0 Off |                    0 |
| N/A   33C    P0             72W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
#  WhatNet  –  Japanese Kuzushiji Hiragana Recognition  (PyTorch)
#  Converted from the TensorFlow/Keras version.
import os, time, random, json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

#  1. CONFIGURATION
CFG = {
    "num_classes":     49,
    "image_size":      28,
    "batch_size":      64,
    "epochs":          100,
    "lr":              5e-4,
    "weight_decay":    1e-4,
    "label_smoothing": 0.1,
    "val_split":       0.1,
    "data_dir":        "/teamspace/studios/this_studio/japanese-dataset/",
    "results_dir":     "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]

#  2. DATASET
def _load_split_files(img_path: str, lbl_path: str):
    images = np.load(img_path)["arr_0"].astype(np.float32)  # (N, 28, 28)
    labels = np.load(lbl_path)["arr_0"].astype(np.int64)
    return images, labels

def _load_combined_file(path: str):
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)
    labels = data["arr_1"].astype(np.int64)
    return images, labels

def _load_kuzushiji(split: str):
    d        = CFG["data_dir"]
    img_path = os.path.join(d, f"k49-{split}-imgs.npz")
    lbl_path = os.path.join(d, f"k49-{split}-labels.npz")
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        print(f"[INFO] Loading {split} from split files.")
        return _load_split_files(img_path, lbl_path)
    combined = os.path.join(d, f"k49_{split}.npz")
    if os.path.exists(combined):
        print(f"[INFO] Loading {split} from combined file.")
        return _load_combined_file(combined)
    raise FileNotFoundError(
        f"[ERROR] Could not find {split} data.\n"
        f"  Checked: {img_path}, {lbl_path}, {combined}"
    )


class KuzushijiDataset(Dataset):
    """Wraps raw numpy arrays; applies transforms at __getitem__ time."""

    def __init__(self, images: np.ndarray, labels: np.ndarray, transform=None):
        # images: (N, 28, 28) float32  labels: (N,) int64
        self.images    = images[:, np.newaxis, :, :]  # → (N, 1, 28, 28)
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx])   # (1, 28, 28)
        lbl = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, lbl


# Normalise: [0,255] → [-1,1]
def normalise(img: torch.Tensor) -> torch.Tensor:
    return img / 127.5 - 1.0


#  Augmentation pipeline (training only)
#   • Brightness / contrast jitter
#   • Pad 2 px → random 28×28 crop  (≈ ±2 px translation)
#   • Random rotation ±8°
#   No horizontal / vertical flip  (asymmetric Kuzushiji glyphs)
train_transform = transforms.Compose([
    transforms.Lambda(normalise),
    transforms.Lambda(lambda x: x.expand(3, -1, -1)),          # need 3-ch for ColorJitter
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Lambda(lambda x: x.mean(0, keepdim=True)),       # back to 1-ch
    transforms.Pad(2, fill=-1.0),
    transforms.RandomCrop(IMG),
    transforms.RandomRotation(degrees=8, fill=-1.0),
])

val_transform = transforms.Compose([
    transforms.Lambda(normalise),
])


# Load & split
x_train_all, y_train_all = _load_kuzushiji("train")
x_test,      y_test      = _load_kuzushiji("test")

print(f"[INFO] Unique classes : {len(np.unique(y_train_all))}")

rng  = np.random.default_rng(SEED)
perm = rng.permutation(len(x_train_all))
x_train_all = x_train_all[perm]
y_train_all = y_train_all[perm]

n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

train_ds   = KuzushijiDataset(x_train, y_train, transform=train_transform)
val_ds     = KuzushijiDataset(x_val,   y_val,   transform=val_transform)
test_ds    = KuzushijiDataset(x_test,  y_test,  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)

#  3. BUILDING BLOCKS

class ResidualBlock(nn.Module):
    """Two 3×3 conv + BN + GELU with identity skip."""
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.net(x) + x)


class DenseResBlock(nn.Module):
    """
    Three chained residual blocks whose outputs are concatenated, then fused
    with a 1×1 conv and downsampled 2× via depthwise-separable strided conv.
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        # projection when channel counts differ
        if in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.GELU(),
            )
        else:
            self.proj = nn.Identity()

        self.r1 = ResidualBlock(out_ch)
        self.r2 = ResidualBlock(out_ch)
        self.r3 = ResidualBlock(out_ch)

        # fuse 3×out_ch → out_ch, then stride-2 depthwise-sep
        self.fuse = nn.Sequential(
            nn.Conv2d(3 * out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )
        self.down = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                      groups=out_ch, bias=False),   # depthwise
            nn.Conv2d(out_ch, out_ch, 1, bias=False),  # pointwise
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        x = self.proj(x)
        r1  = self.r1(x)
        r2  = self.r2(r1)
        r3  = self.r3(r2)
        cat = torch.cat([r1, r2, r3], dim=1)
        out = self.fuse(cat)
        out = self.down(out)
        return out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, C, H, W)
        w = x.mean(dim=[2, 3])         # GAP → (B, C)
        w = self.fc(w).unsqueeze(-1).unsqueeze(-1)   # (B, C, 1, 1)
        return x * w


class AdaptiveFilterCapsule(nn.Module):
    """
    Maps a flat feature vector to K capsule scores, one per class.
    Each capsule learns a class-specific filter over a low-dim projection.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 16):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.h = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Linear(256, num_classes * capsule_dim),
        )
        self.x_proj = nn.Linear(in_features, capsule_dim)
        self.bn     = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        B  = x.size(0)
        h  = self.h(x).view(B, self.num_classes, self.capsule_dim)   # (B,K,D)
        xp = self.x_proj(x).unsqueeze(1).expand_as(h)                # (B,K,D)
        caps = (h * xp).sum(dim=-1)                                   # (B,K)
        caps = self.bn(caps)
        return caps

#  4. MODEL

class WhatNetKuzushiji(nn.Module):
    """
    WhatNet adapted for Kuzushiji-49 Hiragana recognition.

    Stem (dual-path):
      • Standard 3×3 conv   – glyph body
      • Scaffold 3×3 conv   – diagonal cursive stroke structure
      → Concatenated, refined with SE channel attention

    Encoder (3 dense-res stages, each ×2 spatial downsampling):
      enc1: 64  → 64   (14×14)
      enc2: 64  → 128  ( 7× 7)
      enc3: 128 → 256  ( 4× 4, approx)
      Weighted scaffold residuals (+0.1) injected at each stage.

    Decoder head:
      • Multi-scale GAP fusion (enc1 || enc2 || enc3)
      • Adaptive Filter Capsule (AFC)
      • Dense head (linear → LN → GELU → logits)
      • Learned 2-way gate blends AFC and dense-head logits
    """

    def __init__(self, num_classes: int = 49, image_size: int = 28,
                 dropout_rate: float = 0.3, head_units: int = 512):
        super().__init__()
        K = num_classes

        # Stem
        self.stem_main = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_scaffold = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_se   = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
        )

        # Scaffold projections (for residual injection)
        self.sc_proj1 = nn.Conv2d(32, 64,  1, bias=False)
        self.sc_proj2 = nn.Conv2d(32, 128, 1, bias=False)
        self.sc_proj3 = nn.Conv2d(32, 256, 1, bias=False)

        # Encoder
        self.enc1 = DenseResBlock(64,  64)
        self.enc2 = DenseResBlock(64,  128)
        self.enc3 = DenseResBlock(128, 256)

        # Decoder head
        fused_dim = 64 + 128 + 256    # 448
        self.afc      = AdaptiveFilterCapsule(fused_dim, K)
        self.head_fc  = nn.Linear(fused_dim, head_units, bias=False)
        self.head_ln  = nn.LayerNorm(head_units)
        self.head_out = nn.Linear(head_units, K)
        self.gate_fc  = nn.Linear(K + K, 2)   # gate over (dense || afc)

    def forward(self, x):
        # x: (B, 1, 28, 28)

        # Stem
        t        = self.stem_main(x)          # (B, 32, 28, 28)
        scaffold = self.stem_scaffold(x)      # (B, 32, 28, 28)
        stem = torch.cat([t, scaffold], dim=1)  # (B, 64, 28, 28)
        stem = self.stem_se(stem)
        stem = self.stem_proj(stem)

        # Encoder with scaffold residuals
        # enc1: (B, 64, 14, 14)
        enc1 = self.enc1(stem)
        sc1  = F.avg_pool2d(self.sc_proj1(scaffold), 2)
        enc1 = enc1 + 0.1 * sc1

        # enc2: (B, 128, 7, 7)
        enc2 = self.enc2(enc1)
        sc2  = F.avg_pool2d(self.sc_proj2(scaffold), 4)
        enc2 = enc2 + 0.1 * sc2

        # enc3: (B, 256, 4, 4)  (28 → 14 → 7 → 3 or 4 depending on padding)
        enc3 = self.enc3(enc2)
        sc3  = F.adaptive_avg_pool2d(self.sc_proj3(scaffold),
                                     enc3.shape[-2:])
        enc3 = enc3 + 0.1 * sc3

        # Multi-scale GAP fusion
        gap1 = enc1.mean(dim=[2, 3])           # (B, 64)
        gap2 = enc2.mean(dim=[2, 3])           # (B, 128)
        gap3 = enc3.mean(dim=[2, 3])           # (B, 256)
        fused = torch.cat([gap1, gap2, gap3], dim=1)  # (B, 448)

        # AFC
        afc_out = self.afc(fused)              # (B, K)

        # Dense head
        h = F.gelu(self.head_ln(self.head_fc(fused)))
        dense_out = self.head_out(h)           # (B, K)

        # Gated fusion
        gate = F.softmax(
            self.gate_fc(torch.cat([dense_out, afc_out], dim=1)),
            dim=1,
        )                                      # (B, 2)
        logits = dense_out * gate[:, 0:1] + afc_out * gate[:, 1:2]
        return logits                          # (B, K)

#  5. LR SCHEDULE  –  Cosine annealing without restarts
def cosine_annealing_lr(optimizer, step: int, total_steps: int, base_lr: float,
                        min_lr: float = 1e-6) -> float:
    cosine = 0.5 * (1.0 + np.cos(np.pi * step / total_steps))
    lr     = max(base_lr * cosine, min_lr)
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr


#  6. DISPLAY UTILITIES
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"


def print_model_summary(model: nn.Module) -> None:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    W         = 62
    title     = f"  {model.__class__.__name__}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*W}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {total-trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))


def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

#  7. TRAINING LOOP

def train_one_epoch(model, loader, optimizer, criterion, scaler,
                    step: int, total_steps: int) -> tuple[float, float, int]:
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)

        lr = cosine_annealing_lr(optimizer, step, total_steps, CFG["lr"])
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                            enabled=DEVICE.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
        step       += 1

    return total_loss / n, correct / n, step


@torch.no_grad()
def evaluate(model, loader, criterion) -> tuple[float, float]:
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        logits     = model(imgs)
        loss       = criterion(logits, lbls)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def compute_macro_f1(model, loader) -> float:
    model.eval()
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for imgs, lbls in loader:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        preds = model(imgs).argmax(1).cpu().numpy()
        lbls  = lbls.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

#  8. MAIN TRAINING RUN

model     = WhatNetKuzushiji(NUM_CLASSES, IMG).to(DEVICE)
print_model_summary(model)

# Label-smoothing cross-entropy (from_logits – no softmax in model)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]

best_val_acc  = 0.0
patience_ctr  = 0
PATIENCE      = 15
ckpt_path     = os.path.join(CFG["results_dir"], "WhatNet-Kuzushiji_best.pt")

step      = 0
all_hist  = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

t_start = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    tr_loss, tr_acc, step = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, step, total_steps)
    val_loss, val_acc     = evaluate(model, val_loader, criterion)

    all_hist["loss"].append(tr_loss)
    all_hist["accuracy"].append(tr_acc)
    all_hist["val_loss"].append(val_loss)
    all_hist["val_accuracy"].append(val_acc)

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]
    print(
        f"  {_c(f'Epoch {epoch:>3}/{CFG["epochs"]}', 'grey')}  "
        f"{_c(f'loss={tr_loss:.4f}', 'white')}  "
        f"{_c(f'acc={tr_acc*100:.2f}%', 'green')}  "
        f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
        f"{_c(f'val_acc={val_acc*100:.2f}%', 'yellow' if val_acc < tr_acc else 'green')}  "
        f"{_c(f'lr={lr_now:.2e}', 'blue')}  "
        f"{_c(f'[{elapsed:.1f}s]', 'grey')}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(_c(f"\n  Early stopping at epoch {epoch} (patience={PATIENCE})", "yellow"))
            break

elapsed_total = time.time() - t_start
print(f"\n  {_c('✔ Done:', 'green', 'bold')} best val acc = "
      f"{_c(f'{best_val_acc*100:.2f}%', 'green')}  "
      f"wall time = {_c(f'{elapsed_total:.0f}s', 'grey')}")

# Reload best checkpoint for evaluation
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

#  9. FINAL TEST-SET EVALUATION
test_loss, test_acc_raw = evaluate(model, test_loader, criterion)
test_acc  = test_acc_raw * 100.0
macro_f1  = compute_macro_f1(model, test_loader)
n_params  = sum(p.numel() for p in model.parameters())

results = {
    "WhatNet-Kuzushiji": {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    n_params,
        "test_loss": round(float(test_loss), 4),
    }
}
print_comparison_table(results)

#  10. PERSIST RESULTS
results_path = os.path.join(CFG["results_dir"], "kuzushiji_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kuzushiji_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {k: [float(v) for v in vs] for k, vs in all_hist.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kuzushiji-49 benchmark complete.\n", "green", "bold"))

[INFO] Using device: cuda
[INFO] Loading train from split files.
[INFO] Loading test from split files.
[INFO] Unique classes : 49
[INFO] Train: 209,129 | Val: 23,236 | Test: 38,547

╔══════════════════════════════════════════════════════════════╗
║  WhatNetKuzushiji  –  Parameter Summary                      ║
╠══════════════════════════════════════════════════════════════╣
║  Trainable params                      :          5,641,665  ║
║  Non-trainable params                  :                  0  ║
║  Total params                          :          5,641,665  ║
╚══════════════════════════════════════════════════════════════╝

══════════════════════════════════════════════════════════════════════
  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49]
══════════════════════════════════════════════════════════════════════



/tmp/ipykernel_2585/2772786010.py:502: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


  Epoch   1/100  loss=1.1429  acc=86.92%  val_loss=1.6405  val_acc=75.76%  lr=5.00e-04  [48.0s]
  Epoch   2/100  loss=0.8694  acc=95.21%  val_loss=1.4878  val_acc=78.63%  lr=5.00e-04  [45.7s]
  Epoch   3/100  loss=0.8307  acc=96.27%  val_loss=1.0481  val_acc=91.47%  lr=4.99e-04  [45.1s]
  Epoch   4/100  loss=0.8102  acc=96.76%  val_loss=1.0891  val_acc=89.45%  lr=4.98e-04  [45.0s]
  Epoch   5/100  loss=0.7968  acc=97.11%  val_loss=1.0514  val_acc=91.75%  lr=4.97e-04  [44.7s]
  Epoch   6/100  loss=0.7876  acc=97.38%  val_loss=1.6063  val_acc=78.75%  lr=4.96e-04  [43.9s]
  Epoch   7/100  loss=0.7793  acc=97.58%  val_loss=1.1615  val_acc=90.76%  lr=4.94e-04  [44.2s]
  Epoch   8/100  loss=0.7725  acc=97.80%  val_loss=1.1732  val_acc=89.02%  lr=4.92e-04  [44.7s]
  Epoch   9/100  loss=0.7677  acc=97.92%  val_loss=1.0758  val_acc=91.69%  lr=4.90e-04  [44.7s]
  Epoch  10/100  loss=0.7631  acc=98.03%  val_loss=0.9838  val_acc=93.74%  lr=4.88e-04  [44.7s]
  Epoch  11/100  loss=0.7582  acc=98.21%

In [ ]:
[INFO] Using device: cuda
[INFO] Loading train from split files.
[INFO] Loading test from split files.
[INFO] Unique classes : 49
[INFO] Train: 209,129 | Val: 23,236 | Test: 38,547

[96m╔══════════════════════════════════════════════════════════════╗[0m
[96m[1m║  WhatNetKuzushiji  –  Parameter Summary                      ║[0m
[96m╠══════════════════════════════════════════════════════════════╣[0m
[92m║  Trainable params                      :          5,641,665  ║[0m
[90m║  Non-trainable params                  :                  0  ║[0m
[1m[97m║  Total params                          :          5,641,665  ║[0m
[96m╚══════════════════════════════════════════════════════════════╝[0m
[96m
══════════════════════════════════════════════════════════════════════[0m
[1m  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49][0m
[96m══════════════════════════════════════════════════════════════════════
[0m
/tmp/ipykernel_2585/2772786010.py:502: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
  [90mEpoch   1/100[0m  [97mloss=1.1429[0m  [92macc=86.92%[0m  [97mval_loss=1.6405[0m  [93mval_acc=75.76%[0m  [94mlr=5.00e-04[0m  [90m[48.0s][0m
  [90mEpoch   2/100[0m  [97mloss=0.8694[0m  [92macc=95.21%[0m  [97mval_loss=1.4878[0m  [93mval_acc=78.63%[0m  [94mlr=5.00e-04[0m  [90m[45.7s][0m
  [90mEpoch   3/100[0m  [97mloss=0.8307[0m  [92macc=96.27%[0m  [97mval_loss=1.0481[0m  [93mval_acc=91.47%[0m  [94mlr=4.99e-04[0m  [90m[45.1s][0m
  [90mEpoch   4/100[0m  [97mloss=0.8102[0m  [92macc=96.76%[0m  [97mval_loss=1.0891[0m  [93mval_acc=89.45%[0m  [94mlr=4.98e-04[0m  [90m[45.0s][0m
  [90mEpoch   5/100[0m  [97mloss=0.7968[0m  [92macc=97.11%[0m  [97mval_loss=1.0514[0m  [93mval_acc=91.75%[0m  [94mlr=4.97e-04[0m  [90m[44.7s][0m
  [90mEpoch   6/100[0m  [97mloss=0.7876[0m  [92macc=97.38%[0m  [97mval_loss=1.6063[0m  [93mval_acc=78.75%[0m  [94mlr=4.96e-04[0m  [90m[43.9s][0m
  [90mEpoch   7/100[0m  [97mloss=0.7793[0m  [92macc=97.58%[0m  [97mval_loss=1.1615[0m  [93mval_acc=90.76%[0m  [94mlr=4.94e-04[0m  [90m[44.2s][0m
  [90mEpoch   8/100[0m  [97mloss=0.7725[0m  [92macc=97.80%[0m  [97mval_loss=1.1732[0m  [93mval_acc=89.02%[0m  [94mlr=4.92e-04[0m  [90m[44.7s][0m
  [90mEpoch   9/100[0m  [97mloss=0.7677[0m  [92macc=97.92%[0m  [97mval_loss=1.0758[0m  [93mval_acc=91.69%[0m  [94mlr=4.90e-04[0m  [90m[44.7s][0m
  [90mEpoch  10/100[0m  [97mloss=0.7631[0m  [92macc=98.03%[0m  [97mval_loss=0.9838[0m  [93mval_acc=93.74%[0m  [94mlr=4.88e-04[0m  [90m[44.7s][0m
  [90mEpoch  11/100[0m  [97mloss=0.7582[0m  [92macc=98.21%[0m  [97mval_loss=0.9275[0m  [93mval_acc=94.59%[0m  [94mlr=4.85e-04[0m  [90m[44.6s][0m
  [90mEpoch  12/100[0m  [97mloss=0.7544[0m  [92macc=98.28%[0m  [97mval_loss=0.9569[0m  [93mval_acc=94.19%[0m  [94mlr=4.82e-04[0m  [90m[44.3s][0m
  [90mEpoch  13/100[0m  [97mloss=0.7515[0m  [92macc=98.41%[0m  [97mval_loss=0.9955[0m  [93mval_acc=93.04%[0m  [94mlr=4.79e-04[0m  [90m[44.6s][0m
  [90mEpoch  14/100[0m  [97mloss=0.7494[0m  [92macc=98.44%[0m  [97mval_loss=1.2282[0m  [93mval_acc=87.77%[0m  [94mlr=4.76e-04[0m  [90m[44.5s][0m
  [90mEpoch  15/100[0m  [97mloss=0.7457[0m  [92macc=98.57%[0m  [97mval_loss=0.9541[0m  [93mval_acc=93.68%[0m  [94mlr=4.73e-04[0m  [90m[44.4s][0m
  [90mEpoch  16/100[0m  [97mloss=0.7441[0m  [92macc=98.60%[0m  [97mval_loss=0.9816[0m  [93mval_acc=92.77%[0m  [94mlr=4.69e-04[0m  [90m[44.5s][0m
  [90mEpoch  17/100[0m  [97mloss=0.7418[0m  [92macc=98.67%[0m  [97mval_loss=1.7163[0m  [93mval_acc=74.01%[0m  [94mlr=4.65e-04[0m  [90m[44.5s][0m
  [90mEpoch  18/100[0m  [97mloss=0.7397[0m  [92macc=98.73%[0m  [97mval_loss=1.2406[0m  [93mval_acc=86.84%[0m  [94mlr=4.61e-04[0m  [90m[44.6s][0m
  [90mEpoch  19/100[0m  [97mloss=0.7376[0m  [92macc=98.81%[0m  [97mval_loss=1.1430[0m  [93mval_acc=89.67%[0m  [94mlr=4.57e-04[0m  [90m[44.6s][0m
  [90mEpoch  20/100[0m  [97mloss=0.7363[0m  [92macc=98.85%[0m  [97mval_loss=1.1784[0m  [93mval_acc=87.72%[0m  [94mlr=4.52e-04[0m  [90m[44.5s][0m
  [90mEpoch  21/100[0m  [97mloss=0.7345[0m  [92macc=98.92%[0m  [97mval_loss=1.1128[0m  [93mval_acc=90.51%[0m  [94mlr=4.48e-04[0m  [90m[44.0s][0m
  [90mEpoch  22/100[0m  [97mloss=0.7330[0m  [92macc=98.96%[0m  [97mval_loss=1.1816[0m  [93mval_acc=88.56%[0m  [94mlr=4.43e-04[0m  [90m[44.8s][0m
  [90mEpoch  23/100[0m  [97mloss=0.7317[0m  [92macc=98.98%[0m  [97mval_loss=1.2799[0m  [93mval_acc=85.36%[0m  [94mlr=4.38e-04[0m  [90m[44.6s][0m
  [90mEpoch  24/100[0m  [97mloss=0.7302[0m  [92macc=99.01%[0m  [97mval_loss=0.9445[0m  [93mval_acc=93.60%[0m  [94mlr=4.32e-04[0m  [90m[44.9s][0m
  [90mEpoch  25/100[0m  [97mloss=0.7294[0m  [92macc=99.06%[0m  [97mval_loss=1.0750[0m  [93mval_acc=91.92%[0m  [94mlr=4.27e-04[0m  [90m[45.6s][0m
  [90mEpoch  26/100[0m  [97mloss=0.7283[0m  [92macc=99.08%[0m  [97mval_loss=1.1174[0m  [93mval_acc=90.16%[0m  [94mlr=4.21e-04[0m  [90m[45.1s][0m
[93m
  Early stopping at epoch 26 (patience=15)[0m

  [92m[1m✔ Done:[0m best val acc = [92m94.59%[0m  wall time = [90m1165s[0m

[96m[1m╔══════════════════════════════════════════════════════════════════════╗[0m
[96m[1m║  FINAL TEST-SET COMPARISON                                           ║[0m
[96m╠════════════════════════╦════════════╦════════════╦════════════╦══════╣[0m
[96m[1m║  Model                 ║     Params ║   Test Acc ║   Macro F1 ║ Loss ║[0m
[96m╠════════════════════════╬════════════╬════════════╬════════════╬══════╣[0m
[92m[1m║★ WhatNet-Kuzushiji     ║ 5,641,665 ║     91.31%║     90.82%║1.041 ║[0m
[96m╚════════════════════════╩════════════╩════════════╩════════════╩══════╝[0m
[92m[1m
  ★  Winner: WhatNet-Kuzushiji  (91.31% test accuracy)
[0m
[INFO] Results   → ./results/kuzushiji_results.json
[INFO] Histories → ./results/kuzushiji_histories.json
[92m[1m
[DONE] Kuzushiji-49 benchmark complete.
[0m

In [4]:
#  WhatNet  –  Japanese Kuzushiji Hiragana Recognition  (PyTorch)
#  Converted from the TensorFlow/Keras version.
import os, time, random, json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

#  1. CONFIGURATION
CFG = {
    "num_classes":     49,
    "image_size":      28,
    "batch_size":      64,
    "epochs":          100,
    "lr":              2e-4,
    "weight_decay":    1e-4,
    "label_smoothing": 0.1,
    "val_split":       0.1,
    "data_dir":        "/teamspace/studios/this_studio/japanese-dataset/",
    "results_dir":     "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]

#  2. DATASET
def _load_split_files(img_path: str, lbl_path: str):
    images = np.load(img_path)["arr_0"].astype(np.float32)  # (N, 28, 28)
    labels = np.load(lbl_path)["arr_0"].astype(np.int64)
    return images, labels

def _load_combined_file(path: str):
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)
    labels = data["arr_1"].astype(np.int64)
    return images, labels

def _load_kuzushiji(split: str):
    d        = CFG["data_dir"]
    img_path = os.path.join(d, f"k49-{split}-imgs.npz")
    lbl_path = os.path.join(d, f"k49-{split}-labels.npz")
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        print(f"[INFO] Loading {split} from split files.")
        return _load_split_files(img_path, lbl_path)
    combined = os.path.join(d, f"k49_{split}.npz")
    if os.path.exists(combined):
        print(f"[INFO] Loading {split} from combined file.")
        return _load_combined_file(combined)
    raise FileNotFoundError(
        f"[ERROR] Could not find {split} data.\n"
        f"  Checked: {img_path}, {lbl_path}, {combined}"
    )


class KuzushijiDataset(Dataset):
    """Wraps raw numpy arrays; applies transforms at __getitem__ time."""

    def __init__(self, images: np.ndarray, labels: np.ndarray, transform=None):
        # images: (N, 28, 28) float32  labels: (N,) int64
        self.images    = images[:, np.newaxis, :, :]  # → (N, 1, 28, 28)
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx])   # (1, 28, 28)
        lbl = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, lbl


# Normalise: [0,255] → [-1,1]
def normalise(img: torch.Tensor) -> torch.Tensor:
    return img / 127.5 - 1.0


#  Augmentation pipeline (training only)
#   • Brightness / contrast jitter
#   • Pad 2 px → random 28×28 crop  (≈ ±2 px translation)
#   • Random rotation ±8°
#   No horizontal / vertical flip  (asymmetric Kuzushiji glyphs)
train_transform = transforms.Compose([
    transforms.Lambda(normalise),
    transforms.Lambda(lambda x: x.expand(3, -1, -1)),          # need 3-ch for ColorJitter
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Lambda(lambda x: x.mean(0, keepdim=True)),       # back to 1-ch
    transforms.Pad(2, fill=-1.0),
    transforms.RandomCrop(IMG),
    transforms.RandomRotation(degrees=8, fill=-1.0),
])

val_transform = transforms.Compose([
    transforms.Lambda(normalise),
])


# Load & split
x_train_all, y_train_all = _load_kuzushiji("train")
x_test,      y_test      = _load_kuzushiji("test")

print(f"[INFO] Unique classes : {len(np.unique(y_train_all))}")

rng  = np.random.default_rng(SEED)
perm = rng.permutation(len(x_train_all))
x_train_all = x_train_all[perm]
y_train_all = y_train_all[perm]

n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

train_ds   = KuzushijiDataset(x_train, y_train, transform=train_transform)
val_ds     = KuzushijiDataset(x_val,   y_val,   transform=val_transform)
test_ds    = KuzushijiDataset(x_test,  y_test,  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)

#  3. BUILDING BLOCKS

class ResidualBlock(nn.Module):
    """Two 3×3 conv + BN + GELU with identity skip."""
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.net(x) + x)


class DenseResBlock(nn.Module):
    """
    Three chained residual blocks whose outputs are concatenated, then fused
    with a 1×1 conv and downsampled 2× via depthwise-separable strided conv.
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        # projection when channel counts differ
        if in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.GELU(),
            )
        else:
            self.proj = nn.Identity()

        self.r1 = ResidualBlock(out_ch)
        self.r2 = ResidualBlock(out_ch)
        self.r3 = ResidualBlock(out_ch)

        # fuse 3×out_ch → out_ch, then stride-2 depthwise-sep
        self.fuse = nn.Sequential(
            nn.Conv2d(3 * out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )
        self.down = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                      groups=out_ch, bias=False),   # depthwise
            nn.Conv2d(out_ch, out_ch, 1, bias=False),  # pointwise
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        x = self.proj(x)
        r1  = self.r1(x)
        r2  = self.r2(r1)
        r3  = self.r3(r2)
        cat = torch.cat([r1, r2, r3], dim=1)
        out = self.fuse(cat)
        out = self.down(out)
        return out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, C, H, W)
        w = x.mean(dim=[2, 3])         # GAP → (B, C)
        w = self.fc(w).unsqueeze(-1).unsqueeze(-1)   # (B, C, 1, 1)
        return x * w


class AdaptiveFilterCapsule(nn.Module):
    """
    Maps a flat feature vector to K capsule scores, one per class.
    Each capsule learns a class-specific filter over a low-dim projection.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 16):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.h = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Linear(256, num_classes * capsule_dim),
        )
        self.x_proj = nn.Linear(in_features, capsule_dim)
        self.bn     = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        B  = x.size(0)
        h  = self.h(x).view(B, self.num_classes, self.capsule_dim)   # (B,K,D)
        xp = self.x_proj(x).unsqueeze(1).expand_as(h)                # (B,K,D)
        caps = (h * xp).sum(dim=-1)                                   # (B,K)
        caps = self.bn(caps)
        return caps

#  4. MODEL

class WhatNetKuzushiji(nn.Module):
    """
    WhatNet adapted for Kuzushiji-49 Hiragana recognition.

    Stem (dual-path):
      • Standard 3×3 conv   – glyph body
      • Scaffold 3×3 conv   – diagonal cursive stroke structure
      → Concatenated, refined with SE channel attention

    Encoder (3 dense-res stages, each ×2 spatial downsampling):
      enc1: 64  → 64   (14×14)
      enc2: 64  → 128  ( 7× 7)
      enc3: 128 → 256  ( 4× 4, approx)
      Weighted scaffold residuals (+0.1) injected at each stage.

    Decoder head:
      • Multi-scale GAP fusion (enc1 || enc2 || enc3)
      • Adaptive Filter Capsule (AFC)
      • Dense head (linear → LN → GELU → logits)
      • Learned 2-way gate blends AFC and dense-head logits
    """

    def __init__(self, num_classes: int = 49, image_size: int = 28,
                 dropout_rate: float = 0.3, head_units: int = 512):
        super().__init__()
        K = num_classes

        # Stem
        self.stem_main = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_scaffold = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_se   = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
        )

        # Scaffold projections (for residual injection)
        self.sc_proj1 = nn.Conv2d(32, 64,  1, bias=False)
        self.sc_proj2 = nn.Conv2d(32, 128, 1, bias=False)
        self.sc_proj3 = nn.Conv2d(32, 256, 1, bias=False)

        # Encoder
        self.enc1 = DenseResBlock(64,  64)
        self.enc2 = DenseResBlock(64,  128)
        self.enc3 = DenseResBlock(128, 256)

        # Decoder head
        fused_dim = 64 + 128 + 256    # 448
        self.afc      = AdaptiveFilterCapsule(fused_dim, K)
        self.head_fc  = nn.Linear(fused_dim, head_units, bias=False)
        self.head_ln  = nn.LayerNorm(head_units)
        self.head_out = nn.Linear(head_units, K)
        self.gate_fc  = nn.Linear(K + K, 2)   # gate over (dense || afc)

    def forward(self, x):
        # x: (B, 1, 28, 28)

        # Stem
        t        = self.stem_main(x)          # (B, 32, 28, 28)
        scaffold = self.stem_scaffold(x)      # (B, 32, 28, 28)
        stem = torch.cat([t, scaffold], dim=1)  # (B, 64, 28, 28)
        stem = self.stem_se(stem)
        stem = self.stem_proj(stem)

        # Encoder with scaffold residuals
        # enc1: (B, 64, 14, 14)
        enc1 = self.enc1(stem)
        sc1  = F.avg_pool2d(self.sc_proj1(scaffold), 2)
        enc1 = enc1 + 0.1 * sc1

        # enc2: (B, 128, 7, 7)
        enc2 = self.enc2(enc1)
        sc2  = F.avg_pool2d(self.sc_proj2(scaffold), 4)
        enc2 = enc2 + 0.1 * sc2

        # enc3: (B, 256, 4, 4)  (28 → 14 → 7 → 3 or 4 depending on padding)
        enc3 = self.enc3(enc2)
        sc3  = F.adaptive_avg_pool2d(self.sc_proj3(scaffold),
                                     enc3.shape[-2:])
        enc3 = enc3 + 0.1 * sc3

        # Multi-scale GAP fusion
        gap1 = enc1.mean(dim=[2, 3])           # (B, 64)
        gap2 = enc2.mean(dim=[2, 3])           # (B, 128)
        gap3 = enc3.mean(dim=[2, 3])           # (B, 256)
        fused = torch.cat([gap1, gap2, gap3], dim=1)  # (B, 448)

        # AFC
        afc_out = self.afc(fused)              # (B, K)

        # Dense head
        h = F.gelu(self.head_ln(self.head_fc(fused)))
        dense_out = self.head_out(h)           # (B, K)

        # Gated fusion
        gate = F.softmax(
            self.gate_fc(torch.cat([dense_out, afc_out], dim=1)),
            dim=1,
        )                                      # (B, 2)
        logits = dense_out * gate[:, 0:1] + afc_out * gate[:, 1:2]
        return logits                          # (B, K)

#  5. LR SCHEDULE  –  Cosine annealing without restarts
def cosine_annealing_lr(optimizer, step: int, total_steps: int, base_lr: float,
                        min_lr: float = 1e-6) -> float:
    cosine = 0.5 * (1.0 + np.cos(np.pi * step / total_steps))
    lr     = max(base_lr * cosine, min_lr)
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr


#  6. DISPLAY UTILITIES
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"


def print_model_summary(model: nn.Module) -> None:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    W         = 62
    title     = f"  {model.__class__.__name__}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*W}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {total-trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))


def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

#  7. TRAINING LOOP

def train_one_epoch(model, loader, optimizer, criterion, scaler,
                    step: int, total_steps: int) -> tuple[float, float, int]:
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)

        lr = cosine_annealing_lr(optimizer, step, total_steps, CFG["lr"])
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                            enabled=DEVICE.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
        step       += 1

    return total_loss / n, correct / n, step


@torch.no_grad()
def evaluate(model, loader, criterion) -> tuple[float, float]:
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        logits     = model(imgs)
        loss       = criterion(logits, lbls)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def compute_macro_f1(model, loader) -> float:
    model.eval()
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for imgs, lbls in loader:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        preds = model(imgs).argmax(1).cpu().numpy()
        lbls  = lbls.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

#  8. MAIN TRAINING RUN

model     = WhatNetKuzushiji(NUM_CLASSES, IMG).to(DEVICE)
print_model_summary(model)

# Label-smoothing cross-entropy (from_logits – no softmax in model)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]

best_val_acc  = 0.0
patience_ctr  = 0
PATIENCE      = 20
ckpt_path     = os.path.join(CFG["results_dir"], "WhatNet-Kuzushiji_best.pt")

step      = 0
all_hist  = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

t_start = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    tr_loss, tr_acc, step = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, step, total_steps)
    val_loss, val_acc     = evaluate(model, val_loader, criterion)

    all_hist["loss"].append(tr_loss)
    all_hist["accuracy"].append(tr_acc)
    all_hist["val_loss"].append(val_loss)
    all_hist["val_accuracy"].append(val_acc)

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]
    print(
        f"  {_c(f'Epoch {epoch:>3}/{CFG["epochs"]}', 'grey')}  "
        f"{_c(f'loss={tr_loss:.4f}', 'white')}  "
        f"{_c(f'acc={tr_acc*100:.2f}%', 'green')}  "
        f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
        f"{_c(f'val_acc={val_acc*100:.2f}%', 'yellow' if val_acc < tr_acc else 'green')}  "
        f"{_c(f'lr={lr_now:.2e}', 'blue')}  "
        f"{_c(f'[{elapsed:.1f}s]', 'grey')}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(_c(f"\n  Early stopping at epoch {epoch} (patience={PATIENCE})", "yellow"))
            break

elapsed_total = time.time() - t_start
print(f"\n  {_c('✔ Done:', 'green', 'bold')} best val acc = "
      f"{_c(f'{best_val_acc*100:.2f}%', 'green')}  "
      f"wall time = {_c(f'{elapsed_total:.0f}s', 'grey')}")

# Reload best checkpoint for evaluation
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

#  9. FINAL TEST-SET EVALUATION
test_loss, test_acc_raw = evaluate(model, test_loader, criterion)
test_acc  = test_acc_raw * 100.0
macro_f1  = compute_macro_f1(model, test_loader)
n_params  = sum(p.numel() for p in model.parameters())

results = {
    "WhatNet-Kuzushiji": {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    n_params,
        "test_loss": round(float(test_loss), 4),
    }
}
print_comparison_table(results)

#  10. PERSIST RESULTS
results_path = os.path.join(CFG["results_dir"], "kuzushiji_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kuzushiji_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {k: [float(v) for v in vs] for k, vs in all_hist.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kuzushiji-49 benchmark complete.\n", "green", "bold"))

[INFO] Using device: cuda
[INFO] Loading train from split files.
[INFO] Loading test from split files.
[INFO] Unique classes : 49
[INFO] Train: 209,129 | Val: 23,236 | Test: 38,547

╔══════════════════════════════════════════════════════════════╗
║  WhatNetKuzushiji  –  Parameter Summary                      ║
╠══════════════════════════════════════════════════════════════╣
║  Trainable params                      :          5,641,665  ║
║  Non-trainable params                  :                  0  ║
║  Total params                          :          5,641,665  ║
╚══════════════════════════════════════════════════════════════╝

══════════════════════════════════════════════════════════════════════
  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49]
══════════════════════════════════════════════════════════════════════



/tmp/ipykernel_2585/513486785.py:502: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


  Epoch   1/100  loss=1.1728  acc=86.88%  val_loss=1.6639  val_acc=70.05%  lr=2.00e-04  [45.2s]
  Epoch   2/100  loss=0.8756  acc=95.03%  val_loss=1.9755  val_acc=67.81%  lr=2.00e-04  [45.4s]
  Epoch   3/100  loss=0.8315  acc=96.15%  val_loss=1.4873  val_acc=80.23%  lr=2.00e-04  [44.9s]
  Epoch   4/100  loss=0.8101  acc=96.70%  val_loss=1.2683  val_acc=86.76%  lr=1.99e-04  [44.6s]
  Epoch   5/100  loss=0.7966  acc=97.05%  val_loss=1.6600  val_acc=76.30%  lr=1.99e-04  [44.5s]
  Epoch   6/100  loss=0.7862  acc=97.37%  val_loss=1.2711  val_acc=85.42%  lr=1.98e-04  [44.7s]
  Epoch   7/100  loss=0.7785  acc=97.55%  val_loss=1.7308  val_acc=74.11%  lr=1.98e-04  [44.9s]
  Epoch   8/100  loss=0.7715  acc=97.77%  val_loss=1.4624  val_acc=80.31%  lr=1.97e-04  [44.5s]
  Epoch   9/100  loss=0.7670  acc=97.88%  val_loss=1.1854  val_acc=89.16%  lr=1.96e-04  [44.9s]
  Epoch  10/100  loss=0.7618  acc=98.03%  val_loss=1.3281  val_acc=86.11%  lr=1.95e-04  [44.7s]
  Epoch  11/100  loss=0.7579  acc=98.14%

In [ ]:
[INFO] Using device: cuda
[INFO] Loading train from split files.
[INFO] Loading test from split files.
[INFO] Unique classes : 49
[INFO] Train: 209,129 | Val: 23,236 | Test: 38,547

[96m╔══════════════════════════════════════════════════════════════╗[0m
[96m[1m║  WhatNetKuzushiji  –  Parameter Summary                      ║[0m
[96m╠══════════════════════════════════════════════════════════════╣[0m
[92m║  Trainable params                      :          5,641,665  ║[0m
[90m║  Non-trainable params                  :                  0  ║[0m
[1m[97m║  Total params                          :          5,641,665  ║[0m
[96m╚══════════════════════════════════════════════════════════════╝[0m
[96m
══════════════════════════════════════════════════════════════════════[0m
[1m  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49][0m
[96m══════════════════════════════════════════════════════════════════════
[0m
/tmp/ipykernel_2585/513486785.py:502: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
  [90mEpoch   1/100[0m  [97mloss=1.1728[0m  [92macc=86.88%[0m  [97mval_loss=1.6639[0m  [93mval_acc=70.05%[0m  [94mlr=2.00e-04[0m  [90m[45.2s][0m
  [90mEpoch   2/100[0m  [97mloss=0.8756[0m  [92macc=95.03%[0m  [97mval_loss=1.9755[0m  [93mval_acc=67.81%[0m  [94mlr=2.00e-04[0m  [90m[45.4s][0m
  [90mEpoch   3/100[0m  [97mloss=0.8315[0m  [92macc=96.15%[0m  [97mval_loss=1.4873[0m  [93mval_acc=80.23%[0m  [94mlr=2.00e-04[0m  [90m[44.9s][0m
  [90mEpoch   4/100[0m  [97mloss=0.8101[0m  [92macc=96.70%[0m  [97mval_loss=1.2683[0m  [93mval_acc=86.76%[0m  [94mlr=1.99e-04[0m  [90m[44.6s][0m
  [90mEpoch   5/100[0m  [97mloss=0.7966[0m  [92macc=97.05%[0m  [97mval_loss=1.6600[0m  [93mval_acc=76.30%[0m  [94mlr=1.99e-04[0m  [90m[44.5s][0m
  [90mEpoch   6/100[0m  [97mloss=0.7862[0m  [92macc=97.37%[0m  [97mval_loss=1.2711[0m  [93mval_acc=85.42%[0m  [94mlr=1.98e-04[0m  [90m[44.7s][0m
  [90mEpoch   7/100[0m  [97mloss=0.7785[0m  [92macc=97.55%[0m  [97mval_loss=1.7308[0m  [93mval_acc=74.11%[0m  [94mlr=1.98e-04[0m  [90m[44.9s][0m
  [90mEpoch   8/100[0m  [97mloss=0.7715[0m  [92macc=97.77%[0m  [97mval_loss=1.4624[0m  [93mval_acc=80.31%[0m  [94mlr=1.97e-04[0m  [90m[44.5s][0m
  [90mEpoch   9/100[0m  [97mloss=0.7670[0m  [92macc=97.88%[0m  [97mval_loss=1.1854[0m  [93mval_acc=89.16%[0m  [94mlr=1.96e-04[0m  [90m[44.9s][0m
  [90mEpoch  10/100[0m  [97mloss=0.7618[0m  [92macc=98.03%[0m  [97mval_loss=1.3281[0m  [93mval_acc=86.11%[0m  [94mlr=1.95e-04[0m  [90m[44.7s][0m
  [90mEpoch  11/100[0m  [97mloss=0.7579[0m  [92macc=98.14%[0m  [97mval_loss=1.1404[0m  [93mval_acc=89.90%[0m  [94mlr=1.94e-04[0m  [90m[45.1s][0m
  [90mEpoch  12/100[0m  [97mloss=0.7538[0m  [92macc=98.25%[0m  [97mval_loss=0.9960[0m  [93mval_acc=91.95%[0m  [94mlr=1.93e-04[0m  [90m[45.3s][0m
  [90mEpoch  13/100[0m  [97mloss=0.7505[0m  [92macc=98.38%[0m  [97mval_loss=0.9854[0m  [93mval_acc=92.99%[0m  [94mlr=1.92e-04[0m  [90m[44.9s][0m
  [90mEpoch  14/100[0m  [97mloss=0.7485[0m  [92macc=98.41%[0m  [97mval_loss=1.3915[0m  [93mval_acc=82.38%[0m  [94mlr=1.90e-04[0m  [90m[45.7s][0m
  [90mEpoch  15/100[0m  [97mloss=0.7454[0m  [92macc=98.52%[0m  [97mval_loss=1.5039[0m  [93mval_acc=79.70%[0m  [94mlr=1.89e-04[0m  [90m[45.3s][0m
  [90mEpoch  16/100[0m  [97mloss=0.7436[0m  [92macc=98.59%[0m  [97mval_loss=1.1078[0m  [93mval_acc=88.14%[0m  [94mlr=1.88e-04[0m  [90m[45.6s][0m
  [90mEpoch  17/100[0m  [97mloss=0.7409[0m  [92macc=98.67%[0m  [97mval_loss=1.2455[0m  [93mval_acc=86.50%[0m  [94mlr=1.86e-04[0m  [90m[46.3s][0m
  [90mEpoch  18/100[0m  [97mloss=0.7398[0m  [92macc=98.68%[0m  [97mval_loss=1.0841[0m  [93mval_acc=91.62%[0m  [94mlr=1.84e-04[0m  [90m[45.7s][0m
  [90mEpoch  19/100[0m  [97mloss=0.7373[0m  [92macc=98.78%[0m  [97mval_loss=1.1142[0m  [93mval_acc=90.14%[0m  [94mlr=1.83e-04[0m  [90m[45.7s][0m
  [90mEpoch  20/100[0m  [97mloss=0.7359[0m  [92macc=98.84%[0m  [97mval_loss=1.2568[0m  [93mval_acc=85.91%[0m  [94mlr=1.81e-04[0m  [90m[45.5s][0m
  [90mEpoch  21/100[0m  [97mloss=0.7345[0m  [92macc=98.86%[0m  [97mval_loss=1.5373[0m  [93mval_acc=80.34%[0m  [94mlr=1.79e-04[0m  [90m[44.9s][0m
  [90mEpoch  22/100[0m  [97mloss=0.7331[0m  [92macc=98.89%[0m  [97mval_loss=1.1034[0m  [93mval_acc=90.99%[0m  [94mlr=1.77e-04[0m  [90m[44.8s][0m
  [90mEpoch  23/100[0m  [97mloss=0.7318[0m  [92macc=98.94%[0m  [97mval_loss=1.2813[0m  [93mval_acc=86.96%[0m  [94mlr=1.75e-04[0m  [90m[44.9s][0m
  [90mEpoch  24/100[0m  [97mloss=0.7301[0m  [92macc=98.99%[0m  [97mval_loss=1.1375[0m  [93mval_acc=88.94%[0m  [94mlr=1.73e-04[0m  [90m[45.0s][0m
  [90mEpoch  25/100[0m  [97mloss=0.7298[0m  [92macc=99.01%[0m  [97mval_loss=1.1446[0m  [93mval_acc=88.94%[0m  [94mlr=1.71e-04[0m  [90m[44.8s][0m
  [90mEpoch  26/100[0m  [97mloss=0.7278[0m  [92macc=99.08%[0m  [97mval_loss=1.1941[0m  [93mval_acc=87.12%[0m  [94mlr=1.68e-04[0m  [90m[44.6s][0m
  [90mEpoch  27/100[0m  [97mloss=0.7278[0m  [92macc=99.07%[0m  [97mval_loss=1.0678[0m  [93mval_acc=90.29%[0m  [94mlr=1.66e-04[0m  [90m[44.4s][0m
  [90mEpoch  28/100[0m  [97mloss=0.7257[0m  [92macc=99.13%[0m  [97mval_loss=1.0398[0m  [93mval_acc=91.29%[0m  [94mlr=1.64e-04[0m  [90m[46.2s][0m
  [90mEpoch  29/100[0m  [97mloss=0.7255[0m  [92macc=99.13%[0m  [97mval_loss=1.1913[0m  [93mval_acc=88.18%[0m  [94mlr=1.61e-04[0m  [90m[44.7s][0m
  [90mEpoch  30/100[0m  [97mloss=0.7243[0m  [92macc=99.18%[0m  [97mval_loss=1.1170[0m  [93mval_acc=89.40%[0m  [94mlr=1.59e-04[0m  [90m[44.7s][0m
  [90mEpoch  31/100[0m  [97mloss=0.7235[0m  [92macc=99.21%[0m  [97mval_loss=1.2876[0m  [93mval_acc=85.11%[0m  [94mlr=1.56e-04[0m  [90m[44.9s][0m
  [90mEpoch  32/100[0m  [97mloss=0.7227[0m  [92macc=99.23%[0m  [97mval_loss=1.5773[0m  [93mval_acc=77.99%[0m  [94mlr=1.54e-04[0m  [90m[45.0s][0m
  [90mEpoch  33/100[0m  [97mloss=0.7220[0m  [92macc=99.27%[0m  [97mval_loss=1.5138[0m  [93mval_acc=79.56%[0m  [94mlr=1.51e-04[0m  [90m[44.7s][0m
[93m
  Early stopping at epoch 33 (patience=20)[0m

  [92m[1m✔ Done:[0m best val acc = [92m92.99%[0m  wall time = [90m1487s[0m

[96m[1m╔══════════════════════════════════════════════════════════════════════╗[0m
[96m[1m║  FINAL TEST-SET COMPARISON                                           ║[0m
[96m╠════════════════════════╦════════════╦════════════╦════════════╦══════╣[0m
[96m[1m║  Model                 ║     Params ║   Test Acc ║   Macro F1 ║ Loss ║[0m
[96m╠════════════════════════╬════════════╬════════════╬════════════╬══════╣[0m
[92m[1m║★ WhatNet-Kuzushiji     ║ 5,641,665 ║     89.83%║     89.87%║1.096 ║[0m
[96m╚════════════════════════╩════════════╩════════════╩════════════╩══════╝[0m
[92m[1m
  ★  Winner: WhatNet-Kuzushiji  (89.83% test accuracy)
[0m
[INFO] Results   → ./results/kuzushiji_results.json
[INFO] Histories → ./results/kuzushiji_histories.json
[92m[1m
[DONE] Kuzushiji-49 benchmark complete.
[0m

In [6]:
#  WhatNet  –  Japanese Kuzushiji Hiragana Recognition  (PyTorch)
#  Converted from the TensorFlow/Keras version.
import os, time, random, json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

#  1. CONFIGURATION
CFG = {
    "num_classes":     49,
    "image_size":      28,
    "batch_size":      64,
    "epochs":          100,
    "lr":              3e-4,
    "weight_decay":    3e-4,
    "label_smoothing": 0.1,
    "val_split":       0.1,
    "data_dir":        "/teamspace/studios/this_studio/japanese-dataset/",
    "results_dir":     "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]

#  2. DATASET
def _load_split_files(img_path: str, lbl_path: str):
    images = np.load(img_path)["arr_0"].astype(np.float32)  # (N, 28, 28)
    labels = np.load(lbl_path)["arr_0"].astype(np.int64)
    return images, labels

def _load_combined_file(path: str):
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)
    labels = data["arr_1"].astype(np.int64)
    return images, labels

def _load_kuzushiji(split: str):
    d        = CFG["data_dir"]
    img_path = os.path.join(d, f"k49-{split}-imgs.npz")
    lbl_path = os.path.join(d, f"k49-{split}-labels.npz")
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        print(f"[INFO] Loading {split} from split files.")
        return _load_split_files(img_path, lbl_path)
    combined = os.path.join(d, f"k49_{split}.npz")
    if os.path.exists(combined):
        print(f"[INFO] Loading {split} from combined file.")
        return _load_combined_file(combined)
    raise FileNotFoundError(
        f"[ERROR] Could not find {split} data.\n"
        f"  Checked: {img_path}, {lbl_path}, {combined}"
    )


class KuzushijiDataset(Dataset):
    """Wraps raw numpy arrays; applies transforms at __getitem__ time."""

    def __init__(self, images: np.ndarray, labels: np.ndarray, transform=None):
        # images: (N, 28, 28) float32  labels: (N,) int64
        self.images    = images[:, np.newaxis, :, :]  # → (N, 1, 28, 28)
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx])   # (1, 28, 28)
        lbl = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, lbl


# Normalise: [0,255] → [-1,1]
def normalise(img: torch.Tensor) -> torch.Tensor:
    return img / 127.5 - 1.0


#  Augmentation pipeline (training only)
#   • Brightness / contrast jitter
#   • Pad 2 px → random 28×28 crop  (≈ ±2 px translation)
#   • Random rotation ±8°
#   No horizontal / vertical flip  (asymmetric Kuzushiji glyphs)
train_transform = transforms.Compose([
    transforms.Lambda(normalise),
    transforms.Lambda(lambda x: x.expand(3, -1, -1)),          # need 3-ch for ColorJitter
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Lambda(lambda x: x.mean(0, keepdim=True)),       # back to 1-ch
    transforms.Pad(2, fill=-1.0),
    transforms.RandomCrop(IMG),
    transforms.RandomRotation(degrees=8, fill=-1.0),
])

val_transform = transforms.Compose([
    transforms.Lambda(normalise),
])


# Load & split
x_train_all, y_train_all = _load_kuzushiji("train")
x_test,      y_test      = _load_kuzushiji("test")

print(f"[INFO] Unique classes : {len(np.unique(y_train_all))}")

rng  = np.random.default_rng(SEED)
perm = rng.permutation(len(x_train_all))
x_train_all = x_train_all[perm]
y_train_all = y_train_all[perm]

n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

train_ds   = KuzushijiDataset(x_train, y_train, transform=train_transform)
val_ds     = KuzushijiDataset(x_val,   y_val,   transform=val_transform)
test_ds    = KuzushijiDataset(x_test,  y_test,  transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)

#  3. BUILDING BLOCKS

class ResidualBlock(nn.Module):
    """Two 3×3 conv + BN + GELU with identity skip."""
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.net(x) + x)


class DenseResBlock(nn.Module):
    """
    Three chained residual blocks whose outputs are concatenated, then fused
    with a 1×1 conv and downsampled 2× via depthwise-separable strided conv.
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        # projection when channel counts differ
        if in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.GELU(),
            )
        else:
            self.proj = nn.Identity()

        self.r1 = ResidualBlock(out_ch)
        self.r2 = ResidualBlock(out_ch)
        self.r3 = ResidualBlock(out_ch)

        # fuse 3×out_ch → out_ch, then stride-2 depthwise-sep
        self.fuse = nn.Sequential(
            nn.Conv2d(3 * out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )
        self.down = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                      groups=out_ch, bias=False),   # depthwise
            nn.Conv2d(out_ch, out_ch, 1, bias=False),  # pointwise
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        x = self.proj(x)
        r1  = self.r1(x)
        r2  = self.r2(r1)
        r3  = self.r3(r2)
        cat = torch.cat([r1, r2, r3], dim=1) #r3
        out = self.fuse(cat)
        out = self.down(out)
        return out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, C, H, W)
        w = x.mean(dim=[2, 3])         # GAP → (B, C)
        w = self.fc(w).unsqueeze(-1).unsqueeze(-1)   # (B, C, 1, 1)
        return x * w


class AdaptiveFilterCapsule(nn.Module):
    """
    Maps a flat feature vector to K capsule scores, one per class.
    Each capsule learns a class-specific filter over a low-dim projection.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 16):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.h = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Linear(256, num_classes * capsule_dim),
        )
        self.x_proj = nn.Linear(in_features, capsule_dim)
        self.bn     = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        B  = x.size(0)
        h  = self.h(x).view(B, self.num_classes, self.capsule_dim)   # (B,K,D)
        xp = self.x_proj(x).unsqueeze(1).expand_as(h)                # (B,K,D)
        caps = (h * xp).sum(dim=-1)                                   # (B,K)
        caps = self.bn(caps)
        return caps

#  4. MODEL

class WhatNetKuzushiji(nn.Module):
    """
    WhatNet adapted for Kuzushiji-49 Hiragana recognition.

    Stem (dual-path):
      • Standard 3×3 conv   – glyph body
      • Scaffold 3×3 conv   – diagonal cursive stroke structure
      → Concatenated, refined with SE channel attention

    Encoder (3 dense-res stages, each ×2 spatial downsampling):
      enc1: 64  → 64   (14×14)
      enc2: 64  → 128  ( 7× 7)
      enc3: 128 → 256  ( 4× 4, approx)
      Weighted scaffold residuals (+0.1) injected at each stage.

    Decoder head:
      • Multi-scale GAP fusion (enc1 || enc2 || enc3)
      • Adaptive Filter Capsule (AFC)
      • Dense head (linear → LN → GELU → logits)
      • Learned 2-way gate blends AFC and dense-head logits
    """

    def __init__(self, num_classes: int = 49, image_size: int = 28,
                 dropout_rate: float = 0.3, head_units: int = 512):
        super().__init__()
        K = num_classes

        # Stem
        self.stem_main = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_scaffold = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_se   = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
        )

        # Scaffold projections (for residual injection)
        self.sc_proj1 = nn.Conv2d(32, 64,  1, bias=False)
        self.sc_proj2 = nn.Conv2d(32, 128, 1, bias=False)
        self.sc_proj3 = nn.Conv2d(32, 256, 1, bias=False)

        # Encoder
        self.enc1 = DenseResBlock(64,  64)
        self.enc2 = DenseResBlock(64,  128)
        self.enc3 = DenseResBlock(128, 256)

        # Decoder head
        fused_dim = 64 + 128 + 256    # 448
        self.afc      = AdaptiveFilterCapsule(fused_dim, K)
        self.head_fc  = nn.Linear(fused_dim, head_units, bias=False)
        self.head_ln  = nn.LayerNorm(head_units)
        self.head_out = nn.Linear(head_units, K)
        self.gate_fc  = nn.Linear(K + K, 2)   # gate over (dense || afc)

    def forward(self, x):
        # x: (B, 1, 28, 28)

        # Stem
        t        = self.stem_main(x)          # (B, 32, 28, 28)
        scaffold = self.stem_scaffold(x)      # (B, 32, 28, 28)
        stem = torch.cat([t, scaffold], dim=1)  # (B, 64, 28, 28)
        stem = self.stem_se(stem)
        stem = self.stem_proj(stem)

        # Encoder with scaffold residuals
        # enc1: (B, 64, 14, 14)
        enc1 = self.enc1(stem)
        sc1  = F.avg_pool2d(self.sc_proj1(scaffold), 2)
        enc1 = enc1 + 0.1 * sc1

        # enc2: (B, 128, 7, 7)
        enc2 = self.enc2(enc1)
        sc2  = F.avg_pool2d(self.sc_proj2(scaffold), 4)
        enc2 = enc2 + 0.1 * sc2

        # enc3: (B, 256, 4, 4)  (28 → 14 → 7 → 3 or 4 depending on padding)
        enc3 = self.enc3(enc2)
        sc3  = F.adaptive_avg_pool2d(self.sc_proj3(scaffold),
                                     enc3.shape[-2:])
        enc3 = enc3 + 0.1 * sc3

        # Multi-scale GAP fusion
        gap1 = enc1.mean(dim=[2, 3])           # (B, 64)
        gap2 = enc2.mean(dim=[2, 3])           # (B, 128)
        gap3 = enc3.mean(dim=[2, 3])           # (B, 256)
        fused = torch.cat([gap1, gap2, gap3], dim=1)  # (B, 448)

        # AFC
        afc_out = self.afc(fused)              # (B, K)

        # Dense head
        h = F.gelu(self.head_ln(self.head_fc(fused)))
        dense_out = self.head_out(h)           # (B, K)

        # Gated fusion
        gate = F.softmax(
            self.gate_fc(torch.cat([dense_out, afc_out], dim=1)),
            dim=1,
        )                                      # (B, 2)
        logits = dense_out * gate[:, 0:1] + afc_out * gate[:, 1:2]
        return logits                          # (B, K)

#  5. LR SCHEDULE  –  Cosine annealing without restarts
def cosine_annealing_lr(optimizer, step: int, total_steps: int, base_lr: float,
                        min_lr: float = 1e-6) -> float:
    cosine = 0.5 * (1.0 + np.cos(np.pi * step / total_steps))
    lr     = max(base_lr * cosine, min_lr)
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr


#  6. DISPLAY UTILITIES
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"


def print_model_summary(model: nn.Module) -> None:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    W         = 62
    title     = f"  {model.__class__.__name__}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*W}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {total-trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))


def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

#  7. TRAINING LOOP

def train_one_epoch(model, loader, optimizer, criterion, scaler,
                    step: int, total_steps: int) -> tuple[float, float, int]:
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)

        lr = cosine_annealing_lr(optimizer, step, total_steps, CFG["lr"])
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                            enabled=DEVICE.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
        step       += 1

    return total_loss / n, correct / n, step


@torch.no_grad()
def evaluate(model, loader, criterion) -> tuple[float, float]:
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        logits     = model(imgs)
        loss       = criterion(logits, lbls)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def compute_macro_f1(model, loader) -> float:
    model.eval()
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for imgs, lbls in loader:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        preds = model(imgs).argmax(1).cpu().numpy()
        lbls  = lbls.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

#  8. MAIN TRAINING RUN

model     = WhatNetKuzushiji(NUM_CLASSES, IMG).to(DEVICE)
print_model_summary(model)

# Label-smoothing cross-entropy (from_logits – no softmax in model)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]

best_val_acc  = 0.0
patience_ctr  = 0
PATIENCE      = 12
ckpt_path     = os.path.join(CFG["results_dir"], "WhatNet-Kuzushiji_best.pt")

step      = 0
all_hist  = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

t_start = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    tr_loss, tr_acc, step = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, step, total_steps)
    val_loss, val_acc     = evaluate(model, val_loader, criterion)

    all_hist["loss"].append(tr_loss)
    all_hist["accuracy"].append(tr_acc)
    all_hist["val_loss"].append(val_loss)
    all_hist["val_accuracy"].append(val_acc)

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]
    print(
        f"  {_c(f'Epoch {epoch:>3}/{CFG["epochs"]}', 'grey')}  "
        f"{_c(f'loss={tr_loss:.4f}', 'white')}  "
        f"{_c(f'acc={tr_acc*100:.2f}%', 'green')}  "
        f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
        f"{_c(f'val_acc={val_acc*100:.2f}%', 'yellow' if val_acc < tr_acc else 'green')}  "
        f"{_c(f'lr={lr_now:.2e}', 'blue')}  "
        f"{_c(f'[{elapsed:.1f}s]', 'grey')}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(_c(f"\n  Early stopping at epoch {epoch} (patience={PATIENCE})", "yellow"))
            break

elapsed_total = time.time() - t_start
print(f"\n  {_c('✔ Done:', 'green', 'bold')} best val acc = "
      f"{_c(f'{best_val_acc*100:.2f}%', 'green')}  "
      f"wall time = {_c(f'{elapsed_total:.0f}s', 'grey')}")

# Reload best checkpoint for evaluation
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

#  9. FINAL TEST-SET EVALUATION
test_loss, test_acc_raw = evaluate(model, test_loader, criterion)
test_acc  = test_acc_raw * 100.0
macro_f1  = compute_macro_f1(model, test_loader)
n_params  = sum(p.numel() for p in model.parameters())

results = {
    "WhatNet-Kuzushiji": {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    n_params,
        "test_loss": round(float(test_loss), 4),
    }
}
print_comparison_table(results)

#  10. PERSIST RESULTS
results_path = os.path.join(CFG["results_dir"], "kuzushiji_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kuzushiji_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {k: [float(v) for v in vs] for k, vs in all_hist.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kuzushiji-49 benchmark complete.\n", "green", "bold"))

[INFO] Using device: cuda
[INFO] Loading train from split files.


[INFO] Loading test from split files.
[INFO] Unique classes : 49
[INFO] Train: 209,129 | Val: 23,236 | Test: 38,547

╔══════════════════════════════════════════════════════════════╗
║  WhatNetKuzushiji  –  Parameter Summary                      ║
╠══════════════════════════════════════════════════════════════╣
║  Trainable params                      :          5,641,665  ║
║  Non-trainable params                  :                  0  ║
║  Total params                          :          5,641,665  ║
╚══════════════════════════════════════════════════════════════╝

══════════════════════════════════════════════════════════════════════
  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49]
══════════════════════════════════════════════════════════════════════



/tmp/ipykernel_2585/2861304740.py:502: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


  Epoch   1/100  loss=1.1874  acc=86.62%  val_loss=1.0971  val_acc=89.09%  lr=3.00e-04  [46.2s]
  Epoch   2/100  loss=0.8806  acc=94.93%  val_loss=1.6563  val_acc=75.37%  lr=3.00e-04  [45.1s]
  Epoch   3/100  loss=0.8326  acc=96.15%  val_loss=1.3143  val_acc=85.10%  lr=2.99e-04  [45.9s]
  Epoch   4/100  loss=0.8101  acc=96.74%  val_loss=1.2450  val_acc=85.33%  lr=2.99e-04  [46.5s]
  Epoch   5/100  loss=0.7956  acc=97.09%  val_loss=1.5220  val_acc=80.18%  lr=2.98e-04  [45.1s]
  Epoch   6/100  loss=0.7864  acc=97.37%  val_loss=1.5485  val_acc=78.53%  lr=2.97e-04  [45.8s]
  Epoch   7/100  loss=0.7780  acc=97.63%  val_loss=1.1391  val_acc=89.13%  lr=2.96e-04  [45.2s]
  Epoch   8/100  loss=0.7718  acc=97.74%  val_loss=1.0502  val_acc=91.05%  lr=2.95e-04  [45.4s]
  Epoch   9/100  loss=0.7665  acc=97.92%  val_loss=1.0732  val_acc=90.14%  lr=2.94e-04  [45.2s]
  Epoch  10/100  loss=0.7616  acc=98.05%  val_loss=1.1577  val_acc=88.32%  lr=2.93e-04  [45.0s]
  Epoch  11/100  loss=0.7579  acc=98.15%

In [ ]:
[INFO] Using device: cuda
[INFO] Loading train from split files.
[INFO] Loading test from split files.
[INFO] Unique classes : 49
[INFO] Train: 209,129 | Val: 23,236 | Test: 38,547

[96m╔══════════════════════════════════════════════════════════════╗[0m
[96m[1m║  WhatNetKuzushiji  –  Parameter Summary                      ║[0m
[96m╠══════════════════════════════════════════════════════════════╣[0m
[92m║  Trainable params                      :          5,641,665  ║[0m
[90m║  Non-trainable params                  :                  0  ║[0m
[1m[97m║  Total params                          :          5,641,665  ║[0m
[96m╚══════════════════════════════════════════════════════════════╝[0m
[96m
══════════════════════════════════════════════════════════════════════[0m
[1m  Starting training: WhatNet-Kuzushiji  [Kuzushiji-49][0m
[96m══════════════════════════════════════════════════════════════════════
[0m
/tmp/ipykernel_2585/2861304740.py:502: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
  [90mEpoch   1/100[0m  [97mloss=1.1874[0m  [92macc=86.62%[0m  [97mval_loss=1.0971[0m  [92mval_acc=89.09%[0m  [94mlr=3.00e-04[0m  [90m[46.2s][0m
  [90mEpoch   2/100[0m  [97mloss=0.8806[0m  [92macc=94.93%[0m  [97mval_loss=1.6563[0m  [93mval_acc=75.37%[0m  [94mlr=3.00e-04[0m  [90m[45.1s][0m
  [90mEpoch   3/100[0m  [97mloss=0.8326[0m  [92macc=96.15%[0m  [97mval_loss=1.3143[0m  [93mval_acc=85.10%[0m  [94mlr=2.99e-04[0m  [90m[45.9s][0m
  [90mEpoch   4/100[0m  [97mloss=0.8101[0m  [92macc=96.74%[0m  [97mval_loss=1.2450[0m  [93mval_acc=85.33%[0m  [94mlr=2.99e-04[0m  [90m[46.5s][0m
  [90mEpoch   5/100[0m  [97mloss=0.7956[0m  [92macc=97.09%[0m  [97mval_loss=1.5220[0m  [93mval_acc=80.18%[0m  [94mlr=2.98e-04[0m  [90m[45.1s][0m
  [90mEpoch   6/100[0m  [97mloss=0.7864[0m  [92macc=97.37%[0m  [97mval_loss=1.5485[0m  [93mval_acc=78.53%[0m  [94mlr=2.97e-04[0m  [90m[45.8s][0m
  [90mEpoch   7/100[0m  [97mloss=0.7780[0m  [92macc=97.63%[0m  [97mval_loss=1.1391[0m  [93mval_acc=89.13%[0m  [94mlr=2.96e-04[0m  [90m[45.2s][0m
  [90mEpoch   8/100[0m  [97mloss=0.7718[0m  [92macc=97.74%[0m  [97mval_loss=1.0502[0m  [93mval_acc=91.05%[0m  [94mlr=2.95e-04[0m  [90m[45.4s][0m
  [90mEpoch   9/100[0m  [97mloss=0.7665[0m  [92macc=97.92%[0m  [97mval_loss=1.0732[0m  [93mval_acc=90.14%[0m  [94mlr=2.94e-04[0m  [90m[45.2s][0m
  [90mEpoch  10/100[0m  [97mloss=0.7616[0m  [92macc=98.05%[0m  [97mval_loss=1.1577[0m  [93mval_acc=88.32%[0m  [94mlr=2.93e-04[0m  [90m[45.0s][0m
  [90mEpoch  11/100[0m  [97mloss=0.7579[0m  [92macc=98.15%[0m  [97mval_loss=1.1383[0m  [93mval_acc=88.30%[0m  [94mlr=2.91e-04[0m  [90m[45.6s][0m
  [90mEpoch  12/100[0m  [97mloss=0.7539[0m  [92macc=98.28%[0m  [97mval_loss=1.0459[0m  [93mval_acc=90.61%[0m  [94mlr=2.89e-04[0m  [90m[45.1s][0m
  [90mEpoch  13/100[0m  [97mloss=0.7508[0m  [92macc=98.38%[0m  [97mval_loss=0.9137[0m  [93mval_acc=94.70%[0m  [94mlr=2.88e-04[0m  [90m[44.9s][0m
  [90mEpoch  14/100[0m  [97mloss=0.7479[0m  [92macc=98.44%[0m  [97mval_loss=1.1291[0m  [93mval_acc=89.00%[0m  [94mlr=2.86e-04[0m  [90m[45.2s][0m
  [90mEpoch  15/100[0m  [97mloss=0.7451[0m  [92macc=98.53%[0m  [97mval_loss=1.1436[0m  [93mval_acc=88.38%[0m  [94mlr=2.84e-04[0m  [90m[45.6s][0m
  [90mEpoch  16/100[0m  [97mloss=0.7438[0m  [92macc=98.57%[0m  [97mval_loss=0.9310[0m  [93mval_acc=93.59%[0m  [94mlr=2.81e-04[0m  [90m[45.2s][0m
  [90mEpoch  17/100[0m  [97mloss=0.7407[0m  [92macc=98.68%[0m  [97mval_loss=1.1416[0m  [93mval_acc=88.75%[0m  [94mlr=2.79e-04[0m  [90m[45.3s][0m
  [90mEpoch  18/100[0m  [97mloss=0.7396[0m  [92macc=98.72%[0m  [97mval_loss=1.2676[0m  [93mval_acc=85.52%[0m  [94mlr=2.77e-04[0m  [90m[45.1s][0m
  [90mEpoch  19/100[0m  [97mloss=0.7369[0m  [92macc=98.82%[0m  [97mval_loss=1.3556[0m  [93mval_acc=81.67%[0m  [94mlr=2.74e-04[0m  [90m[45.0s][0m
  [90mEpoch  20/100[0m  [97mloss=0.7354[0m  [92macc=98.84%[0m  [97mval_loss=1.7228[0m  [93mval_acc=70.33%[0m  [94mlr=2.71e-04[0m  [90m[45.0s][0m
  [90mEpoch  21/100[0m  [97mloss=0.7337[0m  [92macc=98.88%[0m  [97mval_loss=1.5047[0m  [93mval_acc=79.88%[0m  [94mlr=2.69e-04[0m  [90m[45.0s][0m
  [90mEpoch  22/100[0m  [97mloss=0.7329[0m  [92macc=98.91%[0m  [97mval_loss=1.4123[0m  [93mval_acc=81.27%[0m  [94mlr=2.66e-04[0m  [90m[45.7s][0m
  [90mEpoch  23/100[0m  [97mloss=0.7309[0m  [92macc=98.98%[0m  [97mval_loss=1.1481[0m  [93mval_acc=88.54%[0m  [94mlr=2.63e-04[0m  [90m[46.4s][0m
  [90mEpoch  24/100[0m  [97mloss=0.7296[0m  [92macc=99.03%[0m  [97mval_loss=1.1517[0m  [93mval_acc=88.04%[0m  [94mlr=2.59e-04[0m  [90m[46.2s][0m
  [90mEpoch  25/100[0m  [97mloss=0.7291[0m  [92macc=99.03%[0m  [97mval_loss=1.4014[0m  [93mval_acc=82.65%[0m  [94mlr=2.56e-04[0m  [90m[45.9s][0m
[93m
  Early stopping at epoch 25 (patience=12)[0m

  [92m[1m✔ Done:[0m best val acc = [92m94.70%[0m  wall time = [90m1137s[0m

[96m[1m╔══════════════════════════════════════════════════════════════════════╗[0m
[96m[1m║  FINAL TEST-SET COMPARISON                                           ║[0m
[96m╠════════════════════════╦════════════╦════════════╦════════════╦══════╣[0m
[96m[1m║  Model                 ║     Params ║   Test Acc ║   Macro F1 ║ Loss ║[0m
[96m╠════════════════════════╬════════════╬════════════╬════════════╬══════╣[0m
[92m[1m║★ WhatNet-Kuzushiji     ║ 5,641,665 ║     91.67%║     91.24%║1.028 ║[0m
[96m╚════════════════════════╩════════════╩════════════╩════════════╩══════╝[0m
[92m[1m
  ★  Winner: WhatNet-Kuzushiji  (91.67% test accuracy)
[0m
[INFO] Results   → ./results/kuzushiji_results.json
[INFO] Histories → ./results/kuzushiji_histories.json
[92m[1m
[DONE] Kuzushiji-49 benchmark complete.
[0m

In [ ]:
#  WhatNet  –  Japanese Kuzushiji Hiragana Recognition  (PyTorch)  v2
#  Improvements over v1:
#    1. Linear LR warmup (5 epochs) before cosine decay  → fixes val oscillation
#    2. Lower base LR (2e-4)                             → more stable convergence
#    3. Increased early-stopping patience (30)           → stops firing too early
#    4. Elastic distortion augmentation                  → cursive stroke variation
#    5. MixUp (α=0.2, 50% of batches)                   → smoother decision boundaries
#    6. Stochastic depth in ResidualBlock (p=0.8)        → implicit depth regularisation
#    7. Label smoothing reduced to 0.05                  → less over-smoothing on 49 fine-grained classes
#    8. Model EMA (decay=0.9999)                         → noise-free checkpoint
#    9. Class-balanced WeightedRandomSampler             → handles K49 class imbalance
#   10. Test-time augmentation (TTA, n=8)                → free accuracy at inference

import os, time, random, json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from copy import deepcopy

# ── optional: albumentations for elastic distortion ──────────────────────────
try:
    import albumentations as A
    HAS_ALBUMENTATION = True
except ImportError:
    HAS_ALBUMENTATION = False
    print("[WARN] albumentations not found. Elastic distortion disabled.")
    print("       Install with: pip install albumentations")

# 0. REPRODUCIBILITY ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

# 1. CONFIGURATION ────────────────────────────────────────────────────────────
CFG = {
    "num_classes":     49,
    "image_size":      28,
    "batch_size":      64,
    "epochs":          120,
    # ── v2 changes ──────────────────────────
    "lr":              2e-4,       # was 5e-4; lower = less oscillation
    "weight_decay":    1e-4,
    "label_smoothing": 0.05,       # was 0.1; reduced for fine-grained 49-class task
    "warmup_epochs":   5,          # new: linear warmup before cosine decay
    "patience":        30,         # was 15; give model more time
    "mixup_alpha":     0.2,        # new: MixUp regularisation
    "mixup_prob":      0.5,        # new: apply MixUp 50% of batches
    "ema_decay":       0.9999,     # new: exponential moving average of weights
    "n_tta":           8,          # new: test-time augmentation views
    # ────────────────────────────────────────
    "val_split":       0.1,
    "data_dir":        "/teamspace/studios/this_studio/japanese-dataset/",
    "results_dir":     "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]

# 2. DATASET ──────────────────────────────────────────────────────────────────
def _load_split_files(img_path: str, lbl_path: str):
    images = np.load(img_path)["arr_0"].astype(np.float32)
    labels = np.load(lbl_path)["arr_0"].astype(np.int64)
    return images, labels

def _load_combined_file(path: str):
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)
    labels = data["arr_1"].astype(np.int64)
    return images, labels

def _load_kuzushiji(split: str):
    d        = CFG["data_dir"]
    img_path = os.path.join(d, f"k49-{split}-imgs.npz")
    lbl_path = os.path.join(d, f"k49-{split}-labels.npz")
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        print(f"[INFO] Loading {split} from split files.")
        return _load_split_files(img_path, lbl_path)
    combined = os.path.join(d, f"k49_{split}.npz")
    if os.path.exists(combined):
        print(f"[INFO] Loading {split} from combined file.")
        return _load_combined_file(combined)
    raise FileNotFoundError(
        f"[ERROR] Could not find {split} data.\n"
        f"  Checked: {img_path}, {lbl_path}, {combined}"
    )


# ── Elastic distortion (albumentations) ──────────────────────────────────────
if HAS_ALBUMENTATION:
    _elastic = A.ElasticTransform(alpha=34, sigma=4, p=0.5)

    def elastic_transform(img_np: np.ndarray) -> np.ndarray:
        """img_np: (H, W) uint8 [0,255]"""
        return _elastic(image=img_np)["image"]
else:
    def elastic_transform(img_np: np.ndarray) -> np.ndarray:
        return img_np


class KuzushijiDataset(Dataset):
    """Wraps raw numpy arrays; applies transforms at __getitem__ time."""

    def __init__(self, images: np.ndarray, labels: np.ndarray,
                 transform=None, use_elastic: bool = False):
        self.images      = images[:, np.newaxis, :, :]  # (N,1,28,28)
        self.labels      = labels
        self.transform   = transform
        self.use_elastic = use_elastic and HAS_ALBUMENTATION

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx, 0]          # (28,28) float32
        lbl = int(self.labels[idx])

        # Elastic distortion (training only, on raw uint8 image)
        if self.use_elastic:
            img_u8 = np.clip(img, 0, 255).astype(np.uint8)
            img_u8 = elastic_transform(img_u8)
            img    = img_u8.astype(np.float32)

        img = torch.from_numpy(img).unsqueeze(0)   # (1,28,28)
        if self.transform:
            img = self.transform(img)
        return img, lbl


# ── Normalise: [0,255] → [-1,1] ──────────────────────────────────────────────
def normalise(img: torch.Tensor) -> torch.Tensor:
    return img / 127.5 - 1.0

# Training augmentation pipeline
#   • Brightness/contrast jitter
#   • Pad 2 px → random 28×28 crop  (≈±2 px translation)
#   • Random rotation ±8°
#   No horizontal/vertical flip (asymmetric Kuzushiji glyphs)
train_transform = transforms.Compose([
    transforms.Lambda(normalise),
    transforms.Lambda(lambda x: x.expand(3, -1, -1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Lambda(lambda x: x.mean(0, keepdim=True)),
    transforms.Pad(2, fill=-1.0),
    transforms.RandomCrop(IMG),
    transforms.RandomRotation(degrees=8, fill=-1.0),
])

val_transform = transforms.Compose([
    transforms.Lambda(normalise),
])

# ── Load & split ──────────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_kuzushiji("train")
x_test,      y_test      = _load_kuzushiji("test")

print(f"[INFO] Unique classes : {len(np.unique(y_train_all))}")

rng  = np.random.default_rng(SEED)
perm = rng.permutation(len(x_train_all))
x_train_all = x_train_all[perm]
y_train_all = y_train_all[perm]

n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

train_ds = KuzushijiDataset(x_train, y_train,
                             transform=train_transform, use_elastic=True)
val_ds   = KuzushijiDataset(x_val,   y_val,   transform=val_transform)
test_ds  = KuzushijiDataset(x_test,  y_test,  transform=val_transform)

# ── Class-balanced WeightedRandomSampler ─────────────────────────────────────
class_counts  = np.bincount(y_train, minlength=NUM_CLASSES).astype(np.float32)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = torch.tensor(class_weights[y_train], dtype=torch.float32)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BS, sampler=sampler,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)

# 3. BUILDING BLOCKS ──────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    """Two 3×3 conv + BN + GELU with identity skip.
    v2: optional stochastic depth (survival_prob < 1.0).
    """
    def __init__(self, channels: int, survival_prob: float = 0.8):
        super().__init__()
        self.survival_prob = survival_prob
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        if self.training and self.survival_prob < 1.0:
            if torch.rand(1).item() > self.survival_prob:
                return self.act(x)                      # skip entire block
        return self.act(self.net(x) + x)


class DenseResBlock(nn.Module):
    """
    Three chained residual blocks whose outputs are concatenated, then fused
    with a 1×1 conv and downsampled 2× via depthwise-separable strided conv.
    v2: passes survival_prob to each ResidualBlock.
    """
    def __init__(self, in_ch: int, out_ch: int, survival_prob: float = 0.8):
        super().__init__()
        if in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.GELU(),
            )
        else:
            self.proj = nn.Identity()

        self.r1 = ResidualBlock(out_ch, survival_prob)
        self.r2 = ResidualBlock(out_ch, survival_prob)
        self.r3 = ResidualBlock(out_ch, survival_prob)

        self.fuse = nn.Sequential(
            nn.Conv2d(3 * out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )
        self.down = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                      groups=out_ch, bias=False),
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        x   = self.proj(x)
        r1  = self.r1(x)
        r2  = self.r2(r1)
        r3  = self.r3(r2)
        cat = torch.cat([r1, r2, r3], dim=1)
        out = self.fuse(cat)
        out = self.down(out)
        return out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = x.mean(dim=[2, 3])
        w = self.fc(w).unsqueeze(-1).unsqueeze(-1)
        return x * w


class AdaptiveFilterCapsule(nn.Module):
    """
    Maps a flat feature vector to K capsule scores, one per class.
    Each capsule learns a class-specific filter over a low-dim projection.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 16):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.h = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Linear(256, num_classes * capsule_dim),
        )
        self.x_proj = nn.Linear(in_features, capsule_dim)
        self.bn     = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        B  = x.size(0)
        h  = self.h(x).view(B, self.num_classes, self.capsule_dim)
        xp = self.x_proj(x).unsqueeze(1).expand_as(h)
        caps = (h * xp).sum(dim=-1)
        caps = self.bn(caps)
        return caps

# 4. MODEL ────────────────────────────────────────────────────────────────────

class WhatNetKuzushiji(nn.Module):
    """
    WhatNet adapted for Kuzushiji-49 Hiragana recognition.
    v2: stochastic depth (survival_prob=0.8) added to all ResidualBlocks.
    """

    def __init__(self, num_classes: int = 49, image_size: int = 28,
                 dropout_rate: float = 0.3, head_units: int = 512,
                 survival_prob: float = 0.8):
        super().__init__()
        K = num_classes

        # Stem
        self.stem_main = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_scaffold = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.stem_se   = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
        )

        # Scaffold projections
        self.sc_proj1 = nn.Conv2d(32, 64,  1, bias=False)
        self.sc_proj2 = nn.Conv2d(32, 128, 1, bias=False)
        self.sc_proj3 = nn.Conv2d(32, 256, 1, bias=False)

        # Encoder (with stochastic depth)
        self.enc1 = DenseResBlock(64,  64,  survival_prob)
        self.enc2 = DenseResBlock(64,  128, survival_prob)
        self.enc3 = DenseResBlock(128, 256, survival_prob)

        # Decoder head
        fused_dim = 64 + 128 + 256    # 448
        self.afc      = AdaptiveFilterCapsule(fused_dim, K)
        self.head_fc  = nn.Linear(fused_dim, head_units, bias=False)
        self.head_ln  = nn.LayerNorm(head_units)
        self.head_out = nn.Linear(head_units, K)
        self.gate_fc  = nn.Linear(K + K, 2)

    def forward(self, x):
        # Stem
        t        = self.stem_main(x)
        scaffold = self.stem_scaffold(x)
        stem     = torch.cat([t, scaffold], dim=1)
        stem     = self.stem_se(stem)
        stem     = self.stem_proj(stem)

        # Encoder with scaffold residuals
        enc1 = self.enc1(stem)
        sc1  = F.avg_pool2d(self.sc_proj1(scaffold), 2)
        enc1 = enc1 + 0.1 * sc1

        enc2 = self.enc2(enc1)
        sc2  = F.avg_pool2d(self.sc_proj2(scaffold), 4)
        enc2 = enc2 + 0.1 * sc2

        enc3 = self.enc3(enc2)
        sc3  = F.adaptive_avg_pool2d(self.sc_proj3(scaffold), enc3.shape[-2:])
        enc3 = enc3 + 0.1 * sc3

        # Multi-scale GAP fusion
        gap1  = enc1.mean(dim=[2, 3])
        gap2  = enc2.mean(dim=[2, 3])
        gap3  = enc3.mean(dim=[2, 3])
        fused = torch.cat([gap1, gap2, gap3], dim=1)

        # AFC
        afc_out = self.afc(fused)

        # Dense head
        h         = F.gelu(self.head_ln(self.head_fc(fused)))
        dense_out = self.head_out(h)

        # Gated fusion
        gate   = F.softmax(
            self.gate_fc(torch.cat([dense_out, afc_out], dim=1)), dim=1)
        logits = dense_out * gate[:, 0:1] + afc_out * gate[:, 1:2]
        return logits

# 5. LR SCHEDULE  –  Linear warmup → cosine annealing ────────────────────────

def get_lr(step: int, warmup_steps: int, total_steps: int,
           base_lr: float, min_lr: float = 1e-6) -> float:
    """Linear warmup for `warmup_steps`, then cosine decay to `min_lr`."""
    if step < warmup_steps:
        lr = base_lr * (step + 1) / warmup_steps
    else:
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        lr       = max(base_lr * 0.5 * (1.0 + np.cos(np.pi * progress)), min_lr)
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr

# 6. MODEL EMA ────────────────────────────────────────────────────────────────

class ModelEMA:
    """Exponential moving average of model weights.
    Use .model for evaluation; it tracks a smoothed version of the live model.
    """
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay = decay
        self.model = deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for ema_p, live_p in zip(self.model.parameters(), model.parameters()):
            ema_p.copy_(ema_p * self.decay + live_p.detach() * (1.0 - self.decay))

# 7. MIXUP ────────────────────────────────────────────────────────────────────

def mixup_batch(imgs: torch.Tensor, lbls: torch.Tensor,
                alpha: float = 0.2) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """Returns mixed images, labels_a, labels_b, lambda."""
    lam   = float(np.random.beta(alpha, alpha))
    idx   = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1.0 - lam) * imgs[idx]
    return mixed, lbls, lbls[idx], lam

def mixup_loss(criterion, logits, lbl_a, lbl_b, lam):
    return lam * criterion(logits, lbl_a) + (1.0 - lam) * criterion(logits, lbl_b)

# 8. DISPLAY UTILITIES ────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"


def print_model_summary(model: nn.Module) -> None:
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    W         = 62
    title     = f"  {model.__class__.__name__}  –  Parameter Summary  (v2)"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*W}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {total-trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))


def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# 9. TRAINING LOOP ────────────────────────────────────────────────────────────

def train_one_epoch(model, ema, loader, optimizer, criterion, scaler,
                    step: int, warmup_steps: int, total_steps: int
                    ) -> tuple[float, float, int]:
    model.train()
    total_loss, correct, n = 0.0, 0, 0

    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)

        # Update LR (warmup → cosine)
        lr = get_lr(step, warmup_steps, total_steps, CFG["lr"])
        optimizer.zero_grad(set_to_none=True)

        # MixUp (applied with probability mixup_prob)
        use_mixup = (random.random() < CFG["mixup_prob"])
        if use_mixup:
            imgs, lbl_a, lbl_b, lam = mixup_batch(imgs, lbls, CFG["mixup_alpha"])

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                            enabled=DEVICE.type == "cuda"):
            logits = model(imgs)
            if use_mixup:
                loss = mixup_loss(criterion, logits, lbl_a, lbl_b, lam)
            else:
                loss = criterion(logits, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        # Update EMA after each optimiser step
        ema.update(model)

        total_loss += loss.item() * imgs.size(0)
        # Accuracy computed on live model (MixUp labels are blended, use argmax on lbls)
        if not use_mixup:
            correct += (logits.argmax(1) == lbls).sum().item()
            n       += imgs.size(0)
        else:
            # Approximate accuracy for display: dominant label
            dominant = lbl_a if lam >= 0.5 else lbl_b
            correct += (logits.argmax(1) == dominant).sum().item()
            n       += imgs.size(0)
        step += 1

    return total_loss / n, correct / n, step


@torch.no_grad()
def evaluate(model, loader, criterion) -> tuple[float, float]:
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        logits     = model(imgs)
        loss       = criterion(logits, lbls)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate_tta(model, loader, n_tta: int = 8) -> float:
    """Test-time augmentation: average logits over n_tta augmented views."""
    model.eval()
    correct, n_total = 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        # Accumulate logits over augmented views
        logits_sum = model(imgs)                       # view 1: original (val_transform)
        for _ in range(n_tta - 1):
            aug_imgs = torch.stack([train_transform(img.cpu()).to(DEVICE)
                                    for img in imgs])
            logits_sum = logits_sum + model(aug_imgs)
        correct  += (logits_sum.argmax(1) == lbls).sum().item()
        n_total  += imgs.size(0)
    return correct / n_total


@torch.no_grad()
def compute_macro_f1(model, loader) -> float:
    model.eval()
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for imgs, lbls in loader:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        preds = model(imgs).argmax(1).cpu().numpy()
        lbls  = lbls.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# 10. MAIN TRAINING RUN ───────────────────────────────────────────────────────

model = WhatNetKuzushiji(NUM_CLASSES, IMG, survival_prob=0.8).to(DEVICE)
ema   = ModelEMA(model, decay=CFG["ema_decay"])
print_model_summary(model)

criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]
warmup_steps    = steps_per_epoch * CFG["warmup_epochs"]

best_val_acc  = 0.0
patience_ctr  = 0
PATIENCE      = CFG["patience"]
ckpt_path     = os.path.join(CFG["results_dir"], "WhatNet-Kuzushiji-v2_best.pt")
ckpt_ema_path = os.path.join(CFG["results_dir"], "WhatNet-Kuzushiji-v2_best_ema.pt")

step     = 0
all_hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting training: WhatNet-Kuzushiji v2  [Kuzushiji-49]", "bold"))
print(_c(f"  Warmup: {CFG['warmup_epochs']} epochs | LR: {CFG['lr']} | "
         f"Patience: {PATIENCE} | EMA: {CFG['ema_decay']}", "grey"))
print(_c(f"{'═'*70}\n", "cyan"))

t_start = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    tr_loss, tr_acc, step = train_one_epoch(
        model, ema, train_loader, optimizer, criterion, scaler,
        step, warmup_steps, total_steps)

    # Evaluate both live model and EMA model on validation set
    val_loss, val_acc     = evaluate(model,     val_loader, criterion)
    _,        val_acc_ema = evaluate(ema.model, val_loader, criterion)

    all_hist["loss"].append(tr_loss)
    all_hist["accuracy"].append(tr_acc)
    all_hist["val_loss"].append(val_loss)
    all_hist["val_accuracy"].append(val_acc_ema)  # track EMA accuracy

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]
    print(
        f"  {_c(f'Epoch {epoch:>3}/{CFG[\"epochs\"]}', 'grey')}  "
        f"{_c(f'loss={tr_loss:.4f}', 'white')}  "
        f"{_c(f'acc={tr_acc*100:.2f}%', 'green')}  "
        f"{_c(f'val={val_acc*100:.2f}%', 'white')}  "
        f"{_c(f'ema={val_acc_ema*100:.2f}%', 'yellow' if val_acc_ema < tr_acc else 'green')}  "
        f"{_c(f'lr={lr_now:.2e}', 'blue')}  "
        f"{_c(f'[{elapsed:.1f}s]', 'grey')}"
    )

    # Save best checkpoint based on EMA val accuracy
    if val_acc_ema > best_val_acc:
        best_val_acc = val_acc_ema
        patience_ctr = 0
        torch.save(model.state_dict(),     ckpt_path)
        torch.save(ema.model.state_dict(), ckpt_ema_path)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(_c(f"\n  Early stopping at epoch {epoch} (patience={PATIENCE})", "yellow"))
            break

elapsed_total = time.time() - t_start
print(f"\n  {_c('✔ Done:', 'green', 'bold')} best EMA val acc = "
      f"{_c(f'{best_val_acc*100:.2f}%', 'green')}  "
      f"wall time = {_c(f'{elapsed_total:.0f}s', 'grey')}")

# Reload best EMA checkpoint for evaluation
ema.model.load_state_dict(torch.load(ckpt_ema_path, map_location=DEVICE))

# 11. FINAL TEST-SET EVALUATION ───────────────────────────────────────────────

print(_c("\n  Evaluating on test set (EMA model, no TTA)...", "grey"))
test_loss, test_acc_raw = evaluate(ema.model, test_loader, criterion)
test_acc   = test_acc_raw * 100.0

print(_c(f"  Evaluating with TTA (n={CFG['n_tta']})...", "grey"))
test_acc_tta = evaluate_tta(ema.model, test_loader, n_tta=CFG["n_tta"]) * 100.0

macro_f1 = compute_macro_f1(ema.model, test_loader)
n_params = sum(p.numel() for p in model.parameters())

results = {
    "WhatNet-v2 (EMA)": {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    n_params,
        "test_loss": round(float(test_loss), 4),
    },
    "WhatNet-v2 (EMA+TTA)": {
        "test_acc":  round(test_acc_tta, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    n_params,
        "test_loss": round(float(test_loss), 4),
    },
}
print_comparison_table(results)

# 12. PERSIST RESULTS ─────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "kuzushiji_results_v2.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "kuzushiji_histories_v2.json")
with open(histories_path, "w") as f:
    json.dump(
        {k: [float(v) for v in vs] for k, vs in all_hist.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Kuzushiji-49 v2 benchmark complete.\n", "green", "bold"))